# Figure 2D Slingshot trajectories and gene expression

Run Part 1 before Part 2. Set the shared project root and Rscript location in the settings for each part. Part 1 builds the model with endpoint priors; Part 2 reads that model and plots gene expression. The two original workflows are copied in order. To return to Part 1 after Part 2, rerun its settings before plotting.


## Part 1 Slingshot model with endpoint priors

Build or load the model used by the expression plots in Part 2. Start from the saved embryonic CSS integration and retain all 7,355 cells, their original annotations and UMAP coordinates. Python calls Rscript directly; R requires slingshot and jsonlite (original versions: R 4.5.1 and slingshot 2.18.0).

Run `getLineages` and `getCurves` on the first ten standardized CSS principal components, then use `embedCurves(df=6)` to project curves onto the original UMAP. Native settings are df=5, reweight=False and reassign=False. The root is CVG neuroblast; endpoint priors are Ia, II, Ib, Ic and Late CVG. Late CVG represents the sampling boundary, and stage labels do not identify exact branching dates.

Save the four selected views under `ROOT` as 600-dpi PNGs. The model and projected curves are stored in `ROOT`. `Slingshot.h5ad` stores metadata, pseudotime, lineage weights, input principal components, original UMAP, curves and parameters. Counts remain in the integration input.

Run this part in order before Part 2. Existing results are reused by default. Set `RECOMPUTE=True` after changing analysis parameters. To redraw this part after running Part 2, first rerun its settings cell, then load the saved results and run the plotting cell. Point appearance and label positions are configured in the plotting section; colors are defined in the settings.


In [ ]:
#1. =========== Paths and shared settings
from pathlib import Path
import os,io,json,hashlib,tempfile,subprocess,shutil
# Replace each ROOT placeholder with the required file or directory path before running.
ROOT=Path(".")
if ROOT.name=='Code_def':ROOT=ROOT.parent
# Uses CSS_3000_Embryonic_Inte_adata.h5ad generated by fig1_Embryonic_integration.ipynb.
INPUT=ROOT
# Set the file or directory path.
RESULT_DIR=ROOT
# Uses Slingshot.h5ad generated by Part 1, Slingshot model fitting.
RESULT_FILE=ROOT
# Uses Slingshot.rds generated by Part 1, Slingshot model fitting.
MODEL_FILE=ROOT
# Set the file or directory path.
FIGURES=ROOT
# Set the executable or tool directory path.
R_SCRIPT=ROOT
SEED,N_THREADS=0,1
# Set the file or directory path.
(ROOT).mkdir(exist_ok=True)
# Set the file or directory path.
_runtime=tempfile.TemporaryDirectory(prefix='slingshot-',dir=ROOT)
WORK=Path(_runtime.name)
# Set the file or directory path.
os.environ['MPLCONFIGDIR']=str(ROOT)
for key in ['OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS']:os.environ[key]=str(N_THREADS)
import numpy as np,pandas as pd,anndata as ad
ad.settings.allow_write_nullable_strings=True
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
from scipy.spatial import cKDTree
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt,matplotlib.patheffects as pe
plt.rcdefaults()  # Reset plotting defaults before applying this figure style.
from matplotlib.lines import Line2D
from IPython.display import display,Image
CELL_COLORS={'CVG neuroblast': '#A05837', 'O-2: delaminating otic/CVG neuroblast': '#004D43', 'O-3: developing CVG neuron (not SGN-specific)': '#CB7E98', 'Undifferentiated / proliferative CVG': '#B903AA', 'Late CVG': '#6B7900', 'Differentiating SGN': '#006FA6', 'Undifferentiated SGN': '#008941', 'Type Ia + Type II precursor candidate': '#BA0900', 'Type Ib + Type Ic precursor candidate': '#A77500', 'Emerging Type Ic': '#3B9700', 'Immature Type Ia SGN': '#00846F', 'Immature Type II SGN': '#B79762', 'Immature Type Ib SGN': '#00489C', 'Immature Type Ic SGN': '#CC0744'}
STAGE_COLORS={'E9.5': '#D157A0', 'E10.5': '#CC0744', 'E11.5': '#3B9700', 'E13.5': '#3B5DFF', 'E14.5': '#D16100', 'E15.5': '#00846F', 'E16.5': '#BC23FF', 'E17.5': '#6A3A4C', 'E18.5': '#9B9700'}
TERMINAL=['Immature Type Ia SGN','Immature Type II SGN','Immature Type Ib SGN','Immature Type Ic SGN']


In [ ]:
#2. =========== Saved integration and final analysis parameters
RECOMPUTE=False           # Set True to refit; otherwise reuse the saved model and data.
N_CSS_PCS=30              # Reproduce the PCA basis used for the selected result
N_TRAJECTORY_PCS=10       # Standardize its first ten components
ROOT_CLUSTER='CVG neuroblast'
END_CLUSTERS=TERMINAL+['Late CVG']
NATIVE_DF,UMAP_DF=5,6     # Principal-curve and official projection smoothness
APPROX_POINTS,MAX_ITERATIONS=200,40
CONVERGENCE,SHRINK=.001,1
REWEIGHT,REASSIGN=False,False
params=dict(root=ROOT_CLUSTER,endpoints=END_CLUSTERS,df=NATIVE_DF,view_df=UMAP_DF,
            approx_points=APPROX_POINTS,maxit=MAX_ITERATIONS,thresh=CONVERGENCE,
            shrink=SHRINK,reweight=REWEIGHT,reassign=REASSIGN,seed=SEED,
            css_pcs=N_CSS_PCS,trajectory_pcs=N_TRAJECTORY_PCS)
with INPUT.open('rb') as handle:input_sha=hashlib.file_digest(handle,'sha256').hexdigest()
a=ad.read_h5ad(INPUT,backed='r')
obs=a.obs.copy()
xy=np.asarray(a.obsm['X_umap']).copy()
if RECOMPUTE or not (RESULT_FILE.exists() and MODEL_FILE.exists()):
    with threadpool_limits(limits=N_THREADS):
        pcs=PCA(n_components=N_CSS_PCS,svd_solver='full',random_state=SEED).fit_transform(a.obsm['X_css'])[:,:N_TRAJECTORY_PCS]
    white10=pcs/pcs.std(axis=0,ddof=1)
a.file.close()


In [ ]:
#3. =========== Native Slingshot model and two reusable result files
OBS_FIELDS=['cell_type_plot', 'stage_standardized', 'leiden', 'study_id', 'library_id', 'GSM', 'original_cell_id', 'source_cell_id', 'original_annotation', 'integration_batch', 'source_review_flag', 'source_rna_mixture_review_flag', 'source_state_evaluation_allowed', 'author_SGN_tree_member']  # Metadata needed for plotting, provenance and identity review
# Set the paths used by the embedded R step.
R_SOURCE = r'''ROOT <- "."  # Set the file paths before running this R step.
suppressPackageStartupMessages(library(slingshot))
suppressPackageStartupMessages(library(jsonlite))
work <- normalizePath(commandArgs(trailingOnly=TRUE)[1],winslash='/',mustWork=TRUE)
# Uses parameters.json generated by the trajectory input preparation step.
cfg <- fromJSON(ROOT)
# Uses obs.csv generated by the Slingshot input preparation step.
obs <- read.csv(ROOT,row.names=1,check.names=FALSE)
# Uses white10.csv generated by the Slingshot input preparation step.
rd <- as.matrix(read.csv(ROOT,row.names=1))
# Uses umap.csv generated by the trajectory input preparation step.
umap <- as.matrix(read.csv(ROOT,row.names=1))
set.seed(cfg$seed)
pto <- getLineages(rd,as.character(obs$cell_type_plot),start.clus=cfg$root,
                   end.clus=cfg$endpoints,dist.method='simple',omega=FALSE,use.median=FALSE)
fit <- getCurves(pto,shrink=cfg$shrink,extend='n',stretch=0,
                 approx_points=cfg$approx_points,maxit=cfg$maxit,thresh=cfg$thresh,
                 smoother='smooth.spline',df=cfg$df,reweight=cfg$reweight,
                 reassign=cfg$reassign,allow.breaks=TRUE,shrink.method='cosine')
view <- embedCurves(fit,umap,approx_points=cfg$approx_points,stretch=0,
                    shrink=cfg$shrink,smoother='smooth.spline',df=cfg$view_df,
                    shrink.method='cosine')
versions <- list(R=as.character(getRversion()),slingshot=as.character(packageVersion('slingshot')))
saveRDS(list(fit=fit,umap_view=view,parameters=cfg,cell_ids=rownames(rd),
             # Uses input_sha.txt generated by Part 1, Slingshot input preparation.
             software=versions,input_sha256=readLines(ROOT)),
        # Uses Slingshot.rds generated by Part 1, Slingshot model fitting.
        ROOT,compress='xz')
# Uses pseudotime.csv generated by the Slingshot R fitting step.
write.csv(slingPseudotime(fit),ROOT,quote=FALSE)
# Uses weights.csv generated by the Slingshot R fitting step.
write.csv(slingCurveWeights(fit),ROOT,quote=FALSE)
for(i in seq_along(slingCurves(fit))){
    c <- slingCurves(fit)[[i]]
    # Uses native/UMAP curve CSV files generated by the Slingshot R fitting step.
    write.csv(c$s[c$ord,,drop=FALSE],ROOT,row.names=FALSE)
    c <- slingCurves(view)[[i]]
    # Uses native/UMAP curve CSV files generated by the Slingshot R fitting step.
    write.csv(c$s[c$ord,,drop=FALSE],ROOT,row.names=FALSE)
}
write_json(list(lineages=unname(slingLineages(fit)),lineage_names=colnames(slingPseudotime(fit)),
                # Uses model.json generated by Part 1, Slingshot R fitting.
                software=versions),ROOT,auto_unbox=TRUE)
'''
if RECOMPUTE or not (RESULT_FILE.exists() and MODEL_FILE.exists()):
    # Uses white10.csv generated by the Slingshot input preparation step.
    pd.DataFrame(white10,index=obs.index).to_csv(ROOT)
    # Uses umap.csv generated by the trajectory input preparation step.
    pd.DataFrame(xy.astype(np.float64),index=obs.index).to_csv(ROOT)
    # Uses obs.csv generated by the Slingshot input preparation step.
    obs[['cell_type_plot']].to_csv(ROOT)
    # Uses parameters.json generated by the trajectory input preparation step.
    (ROOT).write_text(json.dumps(params),encoding='utf8')
    # Uses input_sha.txt generated by Part 1, Slingshot input preparation.
    (ROOT).write_text(input_sha+'\n',encoding='utf8')
    # Set the file or directory path.
    (ROOT).write_text(R_SOURCE,encoding='utf8')
    env=os.environ.copy();env.update(LC_ALL='C',TMP=str(WORK),TEMP=str(WORK))
    # Set the file or directory path.
    completed=subprocess.run([R_SCRIPT,str(ROOT),str(WORK)],capture_output=True,text=True,env=env)
    if completed.returncode:raise RuntimeError(completed.stderr)
    # Uses model.json generated by Part 1, Slingshot R fitting.
    info=json.loads((ROOT).read_text())
    result=ad.AnnData(obs=obs[OBS_FIELDS].copy())
    result.obsm['X_umap']=xy
    result.obsm['X_slingshot_input']=white10
    for name in ['pseudotime','weights']:
        # Set the file or directory path.
        table=pd.read_csv(ROOT,index_col=0)
        assert table.index.astype(str).equals(obs.index.astype(str))
        result.obsm['slingshot_'+name]=table.to_numpy()
    result.uns['slingshot']=dict(parameters_json=json.dumps(params,sort_keys=True),
        input_file=str(INPUT.relative_to(ROOT)),input_sha256=input_sha,
        lineage_names=np.asarray(info['lineage_names']),
        lineages={key:np.asarray(path) for key,path in zip(info['lineage_names'],info['lineages'])},
        # Uses native/UMAP curve CSV files generated by the Slingshot R fitting step.
        curves_native=np.stack([pd.read_csv(ROOT).to_numpy() for i in range(1,len(info['lineages'])+1)]),
        # Uses native/UMAP curve CSV files generated by the Slingshot R fitting step.
        curves_umap=np.stack([pd.read_csv(ROOT).to_numpy() for i in range(1,len(info['lineages'])+1)]),
        palettes_json=json.dumps(dict(cell=CELL_COLORS,stage=STAGE_COLORS)),
        software_json=json.dumps(info['software']),model_file=MODEL_FILE.name,
        endpoint_priors=True,full_biological_acceptance=False)
    RESULT_DIR.mkdir(parents=True,exist_ok=True)
    # Uses Slingshot.h5ad generated by Part 1, Slingshot model fitting.
    result.write_h5ad(ROOT,compression='gzip',convert_strings_to_categoricals=False)
    # Copy the results without inheriting temporary-file permissions.
    # Uses Slingshot.h5ad generated by Part 1, Slingshot model fitting.
    shutil.copyfile(ROOT,RESULT_FILE)
    # Uses Slingshot.rds generated by Part 1, Slingshot model fitting.
    shutil.copyfile(ROOT,MODEL_FILE)
else:
    result=ad.read_h5ad(RESULT_FILE)
    saved=result.uns['slingshot']
    assert saved['input_sha256']==input_sha and saved['parameters_json']==json.dumps(params,sort_keys=True),'Input or parameters changed; set RECOMPUTE=True.'
print(f'Slingshot ready: {result.n_obs:,} cells, {len(result.uns["slingshot"]["lineage_names"])} lineages, two reusable files.')


In [ ]:
#4. =========== Load only the saved results for plotting
result=ad.read_h5ad(RESULT_FILE)
obs=result.obs
xy=result.obsm['X_umap']
saved=result.uns['slingshot']
model=dict(lineages=[list(saved['lineages'][key]) for key in saved['lineage_names']],
           curves=list(saved['curves_umap']))


In [ ]:
#5. =========== Four selected figures with the original appearance
POINT_SIZE=2.2
POINT_ALPHA=.72
CURVE_WIDTH=1.05
LABEL_SIZE=9.0
LEGEND_SIZE=8.0
PANEL_SIZE=7.8
DPI=600
INLINE_DPI=140
X_LIMITS=(-7.0,19.0)
Y_LIMITS=(-9.0,17.0)

CELL_TEXT={
 'CVG neuroblast':'CVG neuroblast',
 'O-2: delaminating otic/CVG neuroblast':'Delaminating\notic/CVG',
 'O-3: developing CVG neuron (not SGN-specific)':'Developing CVG',
 'Undifferentiated / proliferative CVG':'Proliferative\nCVG',
 'Late CVG':'Late CVG',
 'Differentiating SGN':'Differentiating SGN',
 'Undifferentiated SGN':'Undifferentiated SGN',
 'Type Ia + Type II precursor candidate':'Ia/II candidate\nbranch',
 'Emerging Type Ic':'Ib/Ic candidate\nbranch',
 'Immature Type Ia SGN':'Immature Ia',
 'Immature Type II SGN':'Immature II',
 'Immature Type Ib SGN':'Immature Ib',
 'Immature Type Ic SGN':'Immature Ic',
}

CELL_LAYOUT={
  'CVG neuroblast':((16.6,2.4),False),
  'O-2: delaminating otic/CVG neuroblast':((13.5,.65),True),
  'O-3: developing CVG neuron (not SGN-specific)':((14.2,-3.9),True),
  'Undifferentiated / proliferative CVG':((16.7,1.0),True),
  'Late CVG':((13.0,-1.45),False),
  'Differentiating SGN':((9.1,-5.9),True),
  'Undifferentiated SGN':((5.7,.2),True),
  'Type Ia + Type II precursor candidate':((2.7,7.6),False),
  'Emerging Type Ic':((-.8,4.35),False),
  'Immature Type Ia SGN':((-1.5,13.55),False),
  'Immature Type II SGN':((4.5,10.1),False),
  'Immature Type Ib SGN':((-4.4,8.7),False),
  'Immature Type Ic SGN':((-3.9,6.0),False),
}
STAGE_POSITIONS={'E9.5': (16.8, 2.0), 'E10.5': (14.3, 0.7), 'E11.5': (18.1, -1.4), 'E13.5_CVG': (13.0, -1.5), 'E13.5_SGN': (9.0, -5.85), 'E14.5': (5.3, -2.5), 'E15.5': (6.0, 1.5), 'E16.5': (2.8, 7.5), 'E17.5': (-2.0, 9.0)}

def state_anchor(mask):
    cloud=xy[np.asarray(mask)]
    return cloud[cKDTree(cloud).query(np.median(cloud,axis=0))[1]]

def make_specs(model,kind):
    terminal={p[-1]:c[-1] for p,c in zip(model['lineages'],model['curves'])}
    if kind=='cell':
        specs=[]
        for key,(position,leader) in CELL_LAYOUT.items():
            anchor=terminal.get(key,state_anchor(obs.cell_type_plot.eq(key)))
            specs.append(dict(text=CELL_TEXT[key],key=key,anchor=list(anchor),position=list(position),leader=leader))
        return specs
    specs=[]
    def add(stage,position,mask,leader=True):
        specs.append(dict(text=stage,key=stage,anchor=list(state_anchor(mask)),position=list(position),leader=leader))
    age=obs.stage_standardized
    add('E9.5',STAGE_POSITIONS['E9.5'],age.eq('E9.5'),False)
    add('E10.5',STAGE_POSITIONS['E10.5'],age.eq('E10.5'))
    add('E11.5',STAGE_POSITIONS['E11.5'],age.eq('E11.5')&obs.cell_type_plot.eq('Undifferentiated / proliferative CVG'))
    specs.append(dict(text='E13.5',key='E13.5_CVG',anchor=list(terminal['Late CVG']),position=list(STAGE_POSITIONS['E13.5_CVG']),leader=False))
    add('E13.5',STAGE_POSITIONS['E13.5_SGN'],age.eq('E13.5')&obs.cell_type_plot.eq('Differentiating SGN'))
    add('E14.5',STAGE_POSITIONS['E14.5'],age.eq('E14.5')&obs.cell_type_plot.eq('Undifferentiated SGN'))
    add('E15.5',STAGE_POSITIONS['E15.5'],age.eq('E15.5'))
    add('E16.5',STAGE_POSITIONS['E16.5'],age.eq('E16.5'))
    add('E17.5',STAGE_POSITIONS['E17.5'],age.eq('E17.5'))
    for state in TERMINAL:
        position=CELL_LAYOUT[state][0]
        specs.append(dict(text='E18.5',key=state,anchor=list(terminal[state]),position=list(position),leader=False))
    return specs

def draw(model,rank,kind,version,output_dir):
    palette=CELL_COLORS if kind=='cell' else STAGE_COLORS
    values=obs.cell_type_plot if kind=='cell' else obs.stage_standardized
    canvas_width=PANEL_SIZE+(1.4 if version=='label' else (5.2 if kind=='cell' else 2.8))
    canvas_height=PANEL_SIZE+1.4
    fig=plt.figure(figsize=(canvas_width,canvas_height))
    ax=fig.add_axes([.72/canvas_width,.75/canvas_height,PANEL_SIZE/canvas_width,PANEL_SIZE/canvas_height])
    for key,color in palette.items():
        mask=values.eq(key).to_numpy()
        ax.scatter(xy[mask,0],xy[mask,1],s=POINT_SIZE,c=color,alpha=POINT_ALPHA,lw=0,zorder=2)
    for curve in model['curves']:
        ax.plot(curve[:,0],curve[:,1],color='.12',lw=CURVE_WIDTH,zorder=3)
    ax.set_xlim(X_LIMITS);ax.set_ylim(Y_LIMITS)
    ax.set_aspect('equal',adjustable='box');ax.set_box_aspect(1)
    ax.set_xlabel('UMAP1',fontsize=11);ax.set_ylabel('UMAP2',fontsize=11)
    ax.tick_params(labelsize=9,direction='out',width=1)
    for side in ['top','right']:ax.spines[side].set_visible(False)
    for side in ['left','bottom']:ax.spines[side].set_linewidth(1.1)
    ax.set_title(f'Slingshot | endpoint priors | result {rank}\n'+('Cell state' if kind=='cell' else 'Developmental stage'),fontsize=11,pad=12)
    if version=='label':
        specs=make_specs(model,kind)
        for spec in specs:
            position=spec['position'];anchor=spec['anchor']
            if spec['leader']:
                ax.annotate(spec['text'],xy=anchor,xytext=position,ha='center',va='center',fontsize=LABEL_SIZE,color='.08',zorder=5,path_effects=[pe.withStroke(linewidth=2.3,foreground='white')],arrowprops=dict(arrowstyle='-',color='.40',lw=.43,alpha=.8,shrinkA=4,shrinkB=1))
            else:
                ax.text(*position,spec['text'],ha='center',va='center',fontsize=LABEL_SIZE,color='.08',zorder=5,path_effects=[pe.withStroke(linewidth=2.3,foreground='white')])
    else:
        handles=[Line2D([],[],marker='o',ls='',color=color,markersize=5,label=f"{key.replace('_',' ')} ({int(values.eq(key).sum()):,})") for key,color in palette.items()]
        ax.legend(handles=handles,loc='center left',bbox_to_anchor=(1.03,.5),frameon=False,fontsize=LEGEND_SIZE,handletextpad=.35,labelspacing=.7)
    # Set the file or directory path.
    path=ROOT
    fig.savefig(path,dpi=DPI,facecolor='white')
    preview=io.BytesIO();fig.savefig(preview,format='png',dpi=INLINE_DPI,facecolor='white')
    plt.close(fig)
    return preview.getvalue()

FIGURES.mkdir(parents=True,exist_ok=True)
for kind in ['cell','stage']:
    for version in ['label','legend']:
        display(Image(data=draw(model,2,kind,version,FIGURES)))
print('Saved the four selected figures at 600 dpi.')
_runtime.cleanup()


## Part 2 Lineage trajectories and gene expression

Use the embryonic CSS RNA object and the Slingshot H5AD/RDS produced in Part 1 to reproduce the 39 lineage and gene-expression figures. Python calls Rscript directly. R requires rhdf5, Matrix, slingshot, mgcv, ragg, magick, jsonlite, openssl and viridisLite; Python requires Pillow, pypdf and ReportLab. No older Genealogy.R or Genealogy.rds is used as the analysis source.

The fixed panel contains 20 reported marker positions and nine candidate transcription-factor positions: Klf6/Sox11 for Ia, Klf6/Bcl11a/Sox11 for II, Klf6 for Ib, Klf6/Sox11 for Ic and Nr2f1 for CVG. These candidates come from prior expression, network-stability and literature review; this notebook does not repeat that selection or treat them as validated drivers.

Keep the original annotations, 7,355 cells, UMAP, trajectories and endpoint priors. Early cells may be shared between lineages. Gray cells show the other lineages as background. Late CVG is a sampling boundary; SGN endpoints are immature subtypes. Pseudotime describes relative progression, not embryonic dates or directly comparable calendar time across lineages.


### Expression models

`SEED=0`. `DISPLAY_K=8` caps the expression GAM basis dimension, `GRID_POINTS=300` sets prediction resolution, `SUPPORT_BINS=12` defines support intervals, and `MIN_BIN_CELLS=5` hides curves where data support is insufficient. Fits use a Tweedie distribution, log link, cubic regression splines and REML to summarize pooled `E[log1p(CP10k)|pseudotime]`, including zero-expression cells. They do not provide biological-replicate confidence intervals or tests of branch differences. Some Klf6 curves are sensitive to smoothing, and CVG has only one source study.

Use saved `X` directly for common genes without renormalization. Expanded markers use raw counts and measurement masks with the complete common-gene library denominator; missing measurements remain NA. Verify IDs, coordinates and input hashes before fitting. No additional cell selection, QC or trajectory reconstruction is performed.


In [ ]:
#1. =========== Paths and shared settings
from pathlib import Path
import json, os, re, shutil, struct, subprocess, tempfile
from PIL import Image, ImageChops
from pypdf import PdfReader, PdfWriter, PageObject, Transformation
from pypdf.generic import ArrayObject, ContentStream, DecodedStreamObject, FloatObject, NameObject, NumberObject
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from io import BytesIO

ROOT = Path(".")
if ROOT.name == "Code_def": ROOT = ROOT.parent
# Uses CSS_3000_Embryonic_Inte_adata.h5ad generated by fig1_Embryonic_integration.ipynb.
INPUT = str(ROOT)
# Uses Slingshot.h5ad generated by Part 1, Slingshot model fitting.
TRAJECTORY = str(ROOT)
# Uses Slingshot.rds generated by Part 1, Slingshot model fitting.
MODEL = str(ROOT)
# Set the file or directory path.
DATA_DIR = ROOT
# Set the file or directory path.
FIG_DIR = ROOT
SEED, DPI = 0, 600
HEADER_HEIGHT, HEADER_FONT = 194, 84  # Pixel equivalents at 600 dpi; shared by both formats.
REDRAW_ONLY = False  # True reuses Genealogy.rds and skips expression fitting.
LINEAGES = {"Ia": 3, "II": 4, "Ib": 1, "Ic": 2, "CVG": 5}
MARKERS = {"Ia": ["Neurod1","Shox2","Calb2","Prox1"], "II": ["Neurod1","Gata3","Th","Plk5"],
           "Ib": ["Neurod1","Runx1","Calb1","Lypd1"], "Ic": ["Neurod1","Pou4f1","Espn","Lypd1"],
           "CVG": ["Neurog1","Neurod1","Insm1","Hes6"]}
TFS = {"Ia": ["Klf6","Sox11"], "II": ["Klf6","Bcl11a","Sox11"], "Ib": ["Klf6"], "Ic": ["Klf6","Sox11"], "CVG": ["Nr2f1"]}
# Set the file or directory path. Set the generated file path.
COMBINED_NAMES = {l: (str(ROOT) if l == "Ic" else str(ROOT)) for l in LINEAGES}
DISPLAY_K, GRID_POINTS, SUPPORT_BINS, MIN_BIN_CELLS = 8, 300, 12, 5
# Set the executable or tool directory path.
RSCRIPT = ROOT
if not RSCRIPT and os.name == "nt":
    import winreg
    with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r"SOFTWARE\R-core\R") as key:
        r_home = Path(winreg.QueryValueEx(key, "InstallPath")[0])
    # Set the executable or tool directory path.
    RSCRIPT = ROOT
if not RSCRIPT: raise RuntimeError("Set RSCRIPT to the installed Rscript executable.")
# Set the file or directory path.
FONT_DIR = ROOT
# Set the file or directory path.
pdfmetrics.registerFont(TTFont("Arial", str(ROOT)))


In [ ]:
# Set the paths used by the embedded R step.
R_SETUP = r'''
options(warn=1)
cfg <- jsonlite::fromJSON(Sys.getenv("GENEALOGY_CONFIG"))
# Set the file or directory path.
ROOT <- cfg$root; INPUT <- ROOT; TRAJECTORY <- ROOT; MODEL <- ROOT
DATA_DIR <- cfg$data_dir; FIG_DIR <- cfg$fig_dir; PDF_DIR <- cfg$pdf_dir
LINEAGES <- unlist(cfg$lineages); MARKERS <- cfg$markers; TFS <- cfg$tfs; COMBINED_NAMES <- unlist(cfg$combined_names)
SEED <- cfg$seed; set.seed(SEED)
'''

# Set the paths used by the embedded R step.
R_ANALYSIS = r'''ROOT <- "."  # Set the file paths before running this R step.

#2. =========== RNA and approved trajectory
suppressPackageStartupMessages({library(rhdf5);library(Matrix);library(slingshot);library(mgcv)})
read_column <- function(file,path) {
    a <- h5readAttributes(file,path)
    if(identical(a[["encoding-type"]],"categorical")) {
        codes <- h5read(file,paste0(path,"/codes")); categories <- h5read(file,paste0(path,"/categories"))
        return(ifelse(codes<0,NA,categories[codes+1L]))
    }
    if(a[["encoding-type"]]%in%c("nullable-string-array","nullable-boolean")) {
        values <- h5read(file,paste0(path,"/values")); mask <- as.logical(as.character(h5read(file,paste0(path,"/mask"))))
        values[mask] <- NA; return(values)
    }
    h5read(file,path)
}
ids <- as.character(read_column(INPUT,"obs/_index")); symbols <- as.character(read_column(INPUT,"var/gene_symbol"))
read_csr <- function(path) new("dgCMatrix",i=as.integer(h5read(INPUT,paste0(path,"/indices"))),p=as.integer(h5read(INPUT,paste0(path,"/indptr"))),
    x=as.numeric(h5read(INPUT,paste0(path,"/data"))),Dim=as.integer(c(length(symbols),length(ids))),Dimnames=list(symbols,ids))
counts <- read_csr("layers/counts"); library_size <- colSums(counts); normalized <- read_csr("X")
fields <- c("study_id","library_id","biological_replicate_id","stage_standardized","cell_type_plot","technology")
obs <- as.data.frame(setNames(lapply(fields,function(k)read_column(INPUT,paste0("obs/",k))),fields)); rownames(obs) <- ids
trajectory <- readRDS(MODEL); ptime <- slingPseudotime(trajectory$fit,na=FALSE); weights <- slingCurveWeights(trajectory$fit)
umap <- t(h5read(TRAJECTORY,"obsm/X_umap")); curves <- aperm(h5read(TRAJECTORY,"uns/slingshot/curves_umap"),c(3,2,1))
stream <- file(INPUT,"rb"); input_sha256 <- as.character(openssl::sha256(stream)); close(stream)
stopifnot(!anyDuplicated(ids),identical(ids,as.character(trajectory$cell_ids)),identical(ids,rownames(ptime)),identical(ids,rownames(weights)),
          identical(ids,as.character(read_column(TRAJECTORY,"obs/_index"))),identical(umap,t(h5read(INPUT,"obsm/X_umap"))),
          input_sha256==as.character(trajectory$input_sha256),all(library_size>0),all(weights%in%c(0,1)),all(is.finite(ptime[weights>0])))
panel <- do.call(rbind,lapply(names(LINEAGES),function(l)data.frame(lineage=l,gene=c(MARKERS[[l]],TFS[[l]]),
    role=c(rep("reported_marker",length(MARKERS[[l]])),rep("potential_TF",length(TFS[[l]]))))))
genes <- unique(panel$gene); extra_symbols <- h5read(INPUT,"uns/marker_gene_symbols")
extra_counts <- h5read(INPUT,"obsm/marker_raw_counts"); extra_measured <- h5read(INPUT,"obsm/marker_measured")
extra_measured <- matrix(as.logical(as.character(extra_measured)),nrow=nrow(extra_counts),ncol=ncol(extra_counts))
stopifnot(all(genes%in%c(symbols,extra_symbols)))
rna <- expression <- matrix(NA_real_,length(genes),length(ids),dimnames=list(genes,ids))
for(g in genes) {
    if(g%in%symbols) { rna[g,] <- as.numeric(counts[g,]); expression[g,] <- as.numeric(normalized[g,]) }
    else { j <- match(g,extra_symbols); rna[g,] <- extra_counts[j,]; rna[g,!extra_measured[j,]] <- NA; expression[g,] <- log1p(rna[g,]/library_size*1e4) }
}

#3. =========== Final expression curves and support
DISPLAY_K <- cfg$display_k; GRID_POINTS <- cfg$grid_points; SUPPORT_BINS <- cfg$support_bins; MIN_BIN_CELLS <- cfg$min_bin_cells
predictions <- expression_models <- support <- summaries <- list()
for(i in seq_len(nrow(panel))) {
    l <- panel$lineage[i]; g <- panel$gene[i]; j <- LINEAGES[l]; key <- paste(l,g)
    ix <- weights[,j]>0 & is.finite(expression[g,]); d <- data.frame(y=expression[g,ix],t=ptime[ix,j])
    m <- gam(y~s(t,k=DISPLAY_K,bs="cr"),data=d,family=tw(link="log"),method="REML")
    grid <- data.frame(t=seq(min(d$t),max(d$t),length.out=GRID_POINTS))
    b <- seq(min(d$t),max(d$t),length.out=SUPPORT_BINS+1L); bin <- findInterval(d$t,b,all.inside=TRUE)
    n <- tabulate(bin,SUPPORT_BINS); pred <- predict(m,grid,type="response")
    pred[n[findInterval(grid$t,b,all.inside=TRUE)]<MIN_BIN_CELLS] <- NA
    stopifnot(m$converged,m$outer.info$conv=="full convergence",all(is.finite(coef(m))),all(is.finite(pred[!is.na(pred)])),all(pred[!is.na(pred)]>=0))
    predictions[[key]] <- data.frame(pseudotime=grid$t,mean_log1pCP10k=pred); expression_models[[key]] <- m
    support[[key]] <- data.frame(left=head(b,-1),right=tail(b,-1),n_cells=n,
        n_positive=sapply(seq_len(SUPPORT_BINS),function(z)sum(d$y[bin==z]>0)),
        n_libraries=sapply(seq_len(SUPPORT_BINS),function(z)length(unique(obs$library_id[ix][bin==z]))))
    summaries[[key]] <- data.frame(lineage=l,gene=g,role=panel$role[i],n_measured=sum(ix),n_lineage=sum(weights[,j]>0),
        n_positive=sum(d$y>0),n_studies=length(unique(obs$study_id[ix])),pseudotime_min=min(d$t),pseudotime_max=max(d$t),
        converged=m$converged,outer_convergence=m$outer.info$conv,edf=sum(m$edf)-1,tweedie_power=m$family$getTheta(TRUE),
        supported_bins=sum(n>=MIN_BIN_CELLS),measurement=ifelse(sum(ix)<sum(weights[,j]>0),"Partial feature coverage; missing is NA","Common gene axis"))
}
summary_table <- do.call(rbind,summaries)
r <- list(ids=ids,obs=obs,counts=rna,expression=expression,library_size=library_size,pseudotime=ptime,weights=weights,umap=umap,curves=curves,
    lineages=LINEAGES,panel=panel,predictions=predictions,expression_models=expression_models,support=support,summary=summary_table,
    parameters=list(seed=SEED,display_k=DISPLAY_K,grid_points=GRID_POINTS,support_bins=SUPPORT_BINS,min_bin_cells=MIN_BIN_CELLS,
        expression_unit="Saved log1p(CP10k); full 11467 common-gene library denominator",method="Pooled Tweedie log GAM, cubic regression spline, REML",
        weights="Frozen 0/1 memberships; shared early cells retained",endpoint_prior=TRUE,late_CVG="Sampling boundary, not mature fate",
        inference="Descriptive curves; no dynamic/branch significance or biological-replicate confidence interval is inferred",trajectory=trajectory$parameters),
    sources=list(rna=cfg$input,trajectory=cfg$trajectory,model=cfg$model,input_sha256=input_sha256),
    software=sapply(c("mgcv","slingshot","rhdf5","ragg"),function(p)as.character(getNamespaceVersion(p))))
dir.create(DATA_DIR,showWarnings=FALSE,recursive=TRUE)
# Uses Genealogy.rds generated by Part 2, RNA expression fitting.
saveRDS(r,ROOT,compress="xz")
# Uses Genealogy.tsv generated by Part 2, RNA expression fitting.
write.table(summary_table,ROOT,sep="\t",quote=FALSE,row.names=FALSE,na="NA")
cat("Frozen cells:",length(ids),"; fitted expression panels:",nrow(panel),"; all converged\n")

'''


### Plotting and vector export

Keep the original canvas, fonts, point appearance, curves and colors. Export 600-dpi PNGs and vector PDFs with Arial text. PDF points are not clipped individually, avoiding thousands of Illustrator clipping masks. Text is organized into semantic objects with embedded fonts and Unicode mappings. Composite PDFs merge vector artwork rather than embedding PNGs.


In [ ]:
# Set the paths used by the embedded R step.
R_DRAW = r'''ROOT <- "."  # Set the file paths before running this R step.

#4. =========== Final single panels and PNG combinations
suppressPackageStartupMessages({library(ragg);library(mgcv)})
PSEUDOTIME_COLORS <- viridisLite::plasma(256L)
open_device <- function(stem,size) {
    if(DRAW_FORMAT=="png") ragg::agg_png(paste0(stem,".png"),width=size[1],height=size[2],units="in",res=PNG_DPI,background="white")
    else {
        # Set the file or directory path.
        output <- ROOT; dir.create(dirname(output),recursive=TRUE,showWarnings=FALSE)
        grDevices::cairo_pdf(output,width=size[1],height=size[2],family="Arial")
    }
}
PNG_DPI <- cfg$dpi
TRAJECTORY_SIZE <- c(4.8,5.2)
TRAJECTORY_POINT_SIZE <- 0.31
TRAJECTORY_POINT_ALPHA <- 0.85
pseudotime_colors <- function(t, limits) {
    index <- 1L + floor((length(PSEUDOTIME_COLORS)-1L)*(t-limits[1])/diff(limits))
    PSEUDOTIME_COLORS[pmax(1L,pmin(length(PSEUDOTIME_COLORS),index))]
}
trajectory_plot <- function(l) {
    j <- LINEAGES[l]; ix <- weights[, j] > 0; limits <- range(ptime[ix,j])
    # Set the file or directory path.
    directory <- ROOT; dir.create(directory, showWarnings=FALSE, recursive=TRUE)
    # Set the file or directory path.
    open_device(ROOT,TRAJECTORY_SIZE)
    layout(matrix(1:2, 2), heights=c(4.15, 1.05))
    par(xpd=if(DRAW_FORMAT=="pdf")NA else FALSE,mar=c(4.7, 3.4, 2.1, 0.8), pty="s", family="sans", las=1, cex=0.95)
    label <- if (l == "CVG") "CVG" else paste("SGN", l)
    plot(umap, type="n", xlim=c(-7,19), ylim=c(-9,17), xaxs="i", yaxs="i", asp=1,
         xlab="UMAP1", ylab="UMAP2", main=paste(label, "| Lineage", j), bty="l")
    points(umap[!ix, ], pch=16, cex=0.28, col=adjustcolor("#C9C9C9", alpha.f=0.55))
    points(umap[ix, ], pch=16, cex=TRAJECTORY_POINT_SIZE, col=adjustcolor(pseudotime_colors(ptime[ix,j],limits), alpha.f=TRAJECTORY_POINT_ALPHA))
    lines(curves[j, , ], lwd=1.7, col="#242424")
    arrows(curves[j, 192, 1], curves[j, 192, 2], curves[j, 200, 1], curves[j, 200, 2], length=0.065, angle=23, lwd=1.7, col="#242424")
    par(xpd=if(DRAW_FORMAT=="pdf")NA else FALSE,mar=c(0, 0, 0, 0), pty="m")
    plot.new(); plot.window(xlim=c(0,1), ylim=c(0,1))
    text(0.5,0.94,"Slingshot pseudotime",cex=0.8)
    edges <- seq(0.22,0.78,length.out=length(PSEUDOTIME_COLORS)+1L)
    rect(head(edges,-1),0.68,tail(edges,-1),0.80,col=PSEUDOTIME_COLORS,border=NA)
    text(c(0.22,0.5,0.78),0.54,sprintf("%.1f",c(limits[1],mean(limits),limits[2])),cex=0.72)
    text(0.5, 0.29, sprintf("Lineage cells: %s; grey: other cells (display only)", format(sum(ix), big.mark=",")), cex=0.7)
    text(0.5, 0.09, if(l=="CVG") "Shared early cells retained; Late CVG = sampled boundary" else "Shared early cells retained; saved trajectory unchanged", cex=0.7)
    dev.off()
}

GENE_SIZE <- c(3.2,2.5)
GENE_POINT_SIZE <- 0.27
GENE_POINT_ALPHA <- 0.55
GENE_LINE_WIDTH <- 1.8
gene_plot <- function(r,l,g) {
    j <- r$lineages[l];ix <- r$weights[,j]>0 & is.finite(r$expression[g,]);t <- r$pseudotime[ix,j];y <- r$expression[g,ix]
    maximum <- max(r$expression[g,],na.rm=TRUE)*1.04
    # Set the file or directory path.
    open_device(ROOT,GENE_SIZE)
    par(xpd=if(DRAW_FORMAT=="pdf")NA else FALSE,mar=c(4.9,4.1,2.3,0.7),mgp=c(2.35,0.7,0),las=1,cex=0.82,family="sans")
    plot(t,y,type="n",xlim=range(t),ylim=c(0,maximum),xaxs="i",yaxs="i",bty="l",xlab="Slingshot pseudotime",ylab="log1p(CP10k)",
         main=if(DRAW_FORMAT=="pdf")bquote(italic(.(g))~.(paste("|",if(l=="CVG")"CVG"else paste("SGN",l)))) else bquote(italic(.(g))~"|"~.(if(l=="CVG")"CVG"else paste("SGN",l))))
    limits <- range(r$pseudotime[r$weights[,j]>0,j]) # Match the trajectory color scale, including partial-coverage genes.
    points(t,y,pch=16,cex=GENE_POINT_SIZE,col=adjustcolor(pseudotime_colors(t,limits),alpha.f=GENE_POINT_ALPHA))
    p <- r$predictions[[paste(l,g)]]
    lines(p$pseudotime,p$mean_log1pCP10k,lwd=GENE_LINE_WIDTH+1,col="white") # Separate the saved curve from dense points.
    segments(head(p$pseudotime,-1),head(p$mean_log1pCP10k,-1),tail(p$pseudotime,-1),tail(p$mean_log1pCP10k,-1),
             lwd=GENE_LINE_WIDTH,col=pseudotime_colors((head(p$pseudotime,-1)+tail(p$pseudotime,-1))/2,limits))
    mtext(sprintf("n=%s/%s cells; studies=%d",format(sum(ix),big.mark=","),format(sum(r$weights[,j]>0),big.mark=","),length(unique(r$obs$study_id[ix]))),side=1,line=3.3,cex=0.64)
    dev.off()
}
COMBINED_HEADER_HEIGHT <- as.integer(round(cfg$header_height*PNG_DPI/600))
COMBINED_HEADER_FONT <- as.integer(round(cfg$header_font*PNG_DPI/600))
combined_plot <- function(r,l) {
    # Set the file or directory path.
    directory <- ROOT
    # Set the file or directory path.
    top <- magick::image_read(ROOT)
    gene_width <- as.integer(GENE_SIZE[1]*PNG_DPI); gene_height <- as.integer(GENE_SIZE[2]*PNG_DPI); width <- gene_width*2L
    # Blank sections inherit the source PNG resolution rather than a default density.
    white <- magick::image_crop(top,"1x1+0+0")
    header <- function(label) magick::image_annotate(magick::image_extent(white,paste0(width,"x",COMBINED_HEADER_HEIGHT),color="white"),
        label,gravity="center",size=COMBINED_HEADER_FONT,color="#444444",font="Arial")
    gene_rows <- function(genes) {
        rows <- lapply(split(genes,ceiling(seq_along(genes)/2L)),function(gs) {
            # Set the file or directory path.
            z <- magick::image_append(do.call(c,lapply(gs,function(g)magick::image_read(ROOT))))
            magick::image_extent(z,paste0(width,"x",gene_height),gravity="center",color="white")
        })
        do.call(c,rows)
    }
    markers <- r$panel$gene[r$panel$lineage==l & r$panel$role=="reported_marker"]
    tfs <- r$panel$gene[r$panel$lineage==l & r$panel$role=="potential_TF"]
    stopifnot(length(markers)==4L,!anyDuplicated(tfs))
    top <- magick::image_extent(top,paste0(width,"x",magick::image_info(top)$height),gravity="center",color="white")
    sections <- c(top,header("Reported markers"),gene_rows(markers))
    sections <- if(length(tfs)) c(sections,header(paste0("Candidate TFs (n=",length(tfs),")")),gene_rows(tfs)) else c(sections,header("No candidate TF passed"))
    combined <- magick::image_append(sections,stack=TRUE)
    # Set the file or directory path.
    output <- ROOT; magick::image_write(combined,path=output,format="png",defines=c("png:exclude-chunk"="tIME"))
}
# Uses Genealogy.rds generated by Part 2, RNA expression fitting.
r <- readRDS(ROOT)
umap <- r$umap; weights <- r$weights; ptime <- r$pseudotime; obs <- r$obs; curves <- r$curves
for(DRAW_FORMAT in c("png","pdf")) {
    for(l in names(LINEAGES)) trajectory_plot(l)
    for(i in seq_len(nrow(r$panel))) gene_plot(r,r$panel$lineage[i],r$panel$gene[i])
}
for(l in names(LINEAGES)) combined_plot(r,l)
cat("Saved PNG panels and native vector sources\n")

'''


In [ ]:
#6. =========== Editable PDF text objects
_FONT_BYTES = {}


def complete_font_tables(data):
    """Keep complete ReportLab subsets; Cairo subsets omit naming tables."""
    if len(data) < 12:
        return False
    count = struct.unpack_from(">H", data, 4)[0]
    if 12 + count*16 > len(data):
        return False
    tags = {data[12+i*16:16+i*16] for i in range(count)}
    return {b"name", b"post", b"OS/2"} <= tags


def finalize_pdf_writer(writer, streams):
    """Finalize the existing vector streams for Illustrator without moving marks."""
    if len(streams) != len(writer.pages):
        raise ValueError("Expected one parsed stream per PDF page.")
    writer.pdf_header = "%PDF-1.7"
    # Set the file or directory path.
    faces = {"ArialMT": str(ROOT), "Arial-BoldMT": str(ROOT),
             # Set the file or directory path.
             "Arial-ItalicMT": str(ROOT), "Arial-BoldItalicMT": str(ROOT)}
    embedded, visited = {}, set()
    report = {"integer_Tr": 0, "removed_groups": 0, "closed_circles": 0,
              "full_fonts": 0, "preserved_fonts": 0}
    for page, stream in zip(writer.pages, streams):
        group = page.get("/Group")
        if group is not None:
            if str(group.get_object().get("/S")) != "/Transparency":
                raise ValueError("Unexpected non-transparency page group.")
            page.pop(NameObject("/Group"))
            report["removed_groups"] += 1
        operations = []
        for operands, operator in stream.operations:
            if operator == b"Tr":
                value = float(operands[0])
                if value != int(value) or not 0 <= value <= 7:
                    raise ValueError("Invalid text rendering mode.")
                operands = [NumberObject(int(value))]
                report["integer_Tr"] += 1
            if (operator in (b"f", b"f*") and len(operations) >= 3
                    and [op for _, op in operations[-3:]] == [b"m", b"c", b"c"]
                    and operations[-3][0][:2] == operations[-1][0][-2:]):
                # Filled two-arc Cairo points already end at their start.
                # Explicit closure also exposes closed paths in Illustrator.
                operations.append(([], b"h"))
                report["closed_circles"] += 1
            operations.append((operands, operator))
        stream.operations = operations
        page.replace_contents(stream)
        for reference in page["/Resources"].get("/Font", {}).values():
            font = reference.get_object()
            if id(font) in visited:
                continue
            visited.add(id(font))
            face = re.sub(r"^[A-Z]{6}\+", "", str(font["/BaseFont"]).lstrip("/"))
            if str(font.get("/Subtype")) != "/TrueType" or face not in faces:
                raise ValueError("Only the reviewed simple Arial TrueType fonts are supported.")
            descriptor = font["/FontDescriptor"].get_object()
            existing = descriptor.get("/FontFile2")
            if existing is not None and complete_font_tables(existing.get_object().get_data()):
                report["preserved_fonts"] += 1
                continue
            if str(font.get("/Encoding")) != "/WinAnsiEncoding":
                raise ValueError("Full-font replacement requires the reviewed WinAnsi code mapping.")
            if face not in embedded:
                if face not in _FONT_BYTES:
                    # Set the file or directory path.
                    _FONT_BYTES[face] = (ROOT).read_bytes()
                data = _FONT_BYTES[face]
                program = DecodedStreamObject()
                program.set_data(data)
                program[NameObject("/Length1")] = NumberObject(len(data))
                embedded[face] = writer._add_object(program.flate_encode())
            font[NameObject("/BaseFont")] = NameObject("/" + face)
            descriptor[NameObject("/FontName")] = NameObject("/" + face)
            descriptor[NameObject("/FontFile2")] = embedded[face]
            report["full_fonts"] += 1
    return report


def normalize_pdf(source, destination):
    """Give each Cairo label one text object and replace Symbol-only spaces."""
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if source == destination:
        raise ValueError("Normalization must write a separate PDF.")
    reader = PdfReader(source, strict=True)
    writer = PdfWriter(clone_from=reader)
    report = {"pages": 0, "original_text_objects": 0, "text_objects": 0,
              "symbol_spaces": 0, "split_gaps": 0}
    streams = []
    identity = (1., 0., 0., 1., 0., 0.)
    defaults = {"font": None, "size": 0., "Tc": 0., "Tw": 0.,
                "Tz": 100., "TL": 0., "Tr": 0., "Ts": 0.}
    state_operators = (b"Tc", b"Tw", b"Tz", b"TL", b"Tr", b"Ts")

    def numbers(values):
        return [FloatObject(float(v)) for v in values]

    def moved(matrix, dx, dy=0.):
        a, b, c, d, e, f = matrix
        return (a, b, c, d, e + a*dx + c*dy, f + b*dx + d*dy)

    def raw_bytes(value):
        return bytes(value) if isinstance(value, bytes) else value.original_bytes

    for page in writer.pages:
        resources = page["/Resources"]
        if any(resources.get(k) for k in ["/XObject", "/Pattern", "/Shading"]):
            raise ValueError("Only reviewed flat Cairo vector PDFs are supported.")
        fonts = resources["/Font"]
        names = {str(k): re.sub(r"^[A-Z]{6}\+", "", str(v.get_object()["/BaseFont"]).lstrip("/"))
                 for k, v in fonts.items()}
        symbols = {k for k, name in names.items() if name.lower().startswith("symbol")}
        regular = [k for k, name in names.items()
                   if name.lower() in {"arial", "arialmt"}]
        if len(regular) != 1:
            raise ValueError("Expected exactly one existing regular Arial font.")
        regular = regular[0]

        def width(font_name, codes):
            font = fonts[font_name].get_object()
            if str(font.get("/Subtype")) not in {"/TrueType", "/Type1"} or "/Widths" not in font:
                raise ValueError("Only explicit simple-font widths are supported.")
            first, widths = int(font["/FirstChar"]), font["/Widths"]
            indices = [code-first for code in codes]
            if any(i < 0 or i >= len(widths) for i in indices):
                raise ValueError("A character is outside its font's width table.")
            return sum(float(widths[i]) for i in indices)

        for name in symbols:
            font = fonts[name].get_object()
            cmap = font.get("/ToUnicode")
            if cmap is None or not re.search(rb"<20>\s*<0020>", cmap.get_object().get_data()):
                raise ValueError("Symbol space lacks an explicit Unicode space mapping.")

        stream = ContentStream(page.get_contents(), writer)
        output, state, stack = [], defaults.copy(), []
        tm, lm, inside, active = identity, identity, False, False

        def font_name():
            return regular if state["font"] in symbols else state["font"]

        def finish():
            nonlocal active
            if active:
                output.append(([], b"ET"))
                report["text_objects"] += 1
                active = False

        def begin():
            nonlocal active
            if active:
                return
            if state["font"] is None or state["size"] <= 0:
                raise ValueError("Text has no valid font state.")
            output.extend([([], b"BT"), (numbers(tm), b"Tm"),
                           ([NameObject(font_name()), FloatObject(state["size"])], b"Tf")])
            output.extend((numbers([state[k.decode()]]), k) for k in state_operators)
            active = True

        def show(value):
            nonlocal tm
            codes = raw_bytes(value)
            if not codes:
                return
            old_width = width(state["font"], codes)
            adjustment = 0.
            if state["font"] in symbols:
                if any(code != 32 for code in codes):
                    raise ValueError("Refusing to substitute a genuine Symbol glyph.")
                adjustment = width(regular, codes) - old_width
                report["symbol_spaces"] += len(codes)
            begin()
            if adjustment:
                output.append(([ArrayObject([value, FloatObject(adjustment)])], b"TJ"))
            else:
                output.append(([value], b"Tj"))
            advance = (old_width/1000*state["size"] + len(codes)*state["Tc"]
                       + codes.count(32)*state["Tw"])*state["Tz"]/100
            tm = moved(tm, advance)

        def offset(value):
            nonlocal tm
            value = float(value)
            if abs(value) >= 500:
                finish()
                report["split_gaps"] += 1
            elif active:
                output.append(([ArrayObject([FloatObject(value)])], b"TJ"))
            tm = moved(tm, -value/1000*state["size"]*state["Tz"]/100)

        for operands, operator in stream.operations:
            if operator == b"BT":
                if inside:
                    raise ValueError("Nested native text object.")
                inside, tm, lm = True, identity, identity
                report["original_text_objects"] += 1
            elif operator == b"ET":
                if not inside:
                    raise ValueError("Unmatched native ET.")
                finish()
                inside = False
            elif operator == b"Tf":
                state["font"], state["size"] = str(operands[0]), float(operands[1])
                if active or not inside:
                    output.append(([NameObject(font_name()), operands[1]], operator))
            elif operator in state_operators:
                state[operator.decode()] = float(operands[0])
                if active or not inside:
                    output.append((operands, operator))
            elif inside:
                if operator == b"Tm":
                    finish()
                    tm = lm = tuple(map(float, operands))
                elif operator in {b"Td", b"TD", b"T*"}:
                    finish()
                    if operator == b"T*":
                        dx, dy = 0., -state["TL"]
                    else:
                        dx, dy = map(float, operands)
                        if operator == b"TD":
                            state["TL"] = -dy
                    tm = lm = moved(lm, dx, dy)
                elif operator == b"Tj":
                    show(operands[0])
                elif operator == b"TJ":
                    for item in operands[0]:
                        offset(item) if isinstance(item, (int, float)) else show(item)
                else:
                    raise ValueError(f"Unreviewed Cairo text operator: {operator!r}")
            else:
                if operator in {b"W", b"W*", b"Do", b"BI", b"INLINE IMAGE", b"sh"}:
                    raise ValueError("Native input is outside the zero-clip flat-vector profile.")
                if operator == b"q":
                    stack.append(state.copy())
                elif operator == b"Q":
                    if not stack:
                        raise ValueError("Unbalanced native graphics state.")
                    state = stack.pop()
                output.append((operands, operator))
        if inside or active or stack:
            raise ValueError("Unclosed native text or graphics state.")
        for name in symbols:
            fonts.pop(NameObject(name))
        stream.operations = output
        streams.append(stream)
        report["pages"] += 1
    report.update(finalize_pdf_writer(writer, streams))
    destination.parent.mkdir(parents=True, exist_ok=True)
    writer.write(destination)
    return report


In [ ]:
#7. =========== Vector lineage combinations
def append_vector_page(target, source, x, y):
    reader = PdfReader(source)
    target.merge_transformed_page(reader.pages[0], Transformation().translate(x,y), over=True)

def combined_pdf(lineage, figure_dir):
    # Set the file or directory path.
    directory = ROOT
    # Set the file or directory path.
    with Image.open(ROOT) as im: gene_width,gene_height = [v/DPI*72 for v in im.size]
    # Set the file or directory path.
    with Image.open(ROOT) as im: top_width,top_height = [v/DPI*72 for v in im.size]
    width,header_height = gene_width*2,HEADER_HEIGHT/600*72
    tfs = TFS[lineage]
    height = top_height + 2*gene_height + 2*header_height + ((len(tfs)+1)//2)*gene_height
    page = PageObject.create_blank_page(width=width, height=height)
    y = height-top_height
    # Set the file or directory path.
    append_vector_page(page,ROOT,(width-top_width)/2,y)
    for label,genes in [("Reported markers",MARKERS[lineage]),(f"Candidate TFs (n={len(tfs)})",tfs)]:
        y -= header_height
        buffer = BytesIO(); c = canvas.Canvas(buffer,pagesize=(width,header_height),initialFontName="Arial")
        c.setFont("Arial",HEADER_FONT/600*72)
        c.setFillColorRGB(68/255,68/255,68/255)
        c.drawCentredString(width/2,header_height/2-HEADER_FONT/600*72*0.35,label)
        c.save();buffer.seek(0);append_vector_page(page,buffer,0,y)
        for i in range(0,len(genes),2):
            y -= gene_height
            pair = genes[i:i+2]
            for column,gene in enumerate(pair):
                x = (width-gene_width)/2 if len(pair)==1 else column*gene_width
                # Set the file or directory path.
                append_vector_page(page,ROOT,x,y)
    writer = PdfWriter();writer.add_page(page)
    # Merging inserts page-box clips. All source marks already lie within their pages.
    stream = ContentStream(writer.pages[0].get_contents(),writer)
    stream.operations = [(a,op) for a,op in stream.operations if op not in {b"W",b"W*"}]
    finalize_pdf_writer(writer, [stream])
    # Set the file or directory path.
    writer.write(ROOT)


### Run the analysis or redraw saved results

Run the next section to rebuild expression curves from the upstream results and export the figures. For appearance changes, set `REDRAW_ONLY=True` and rerun the settings, plotting definitions and execution sections to reuse `ROOT`. Changes to `DISPLAY_K` or support rules require the full expression analysis.

Save two main result files: `Genealogy.rds` contains expression, counts, IDs, metadata, fixed trajectories, pseudotime, weights, panels, models, predictions and support; `Genealogy.tsv` summarizes the 29 panel positions and their fits and measurements. Temporary PDFs are cleaned automatically. The earlier genealogy results and Slingshot inputs remain unchanged.


In [ ]:
#8. =========== Run from integration or redraw from the compact RDS
def run_workflow():
    DATA_DIR.mkdir(parents=True,exist_ok=True)
    # Set the temporary parent directory; this run gets its own removable folder.
    work = Path(tempfile.mkdtemp(prefix="genealogy-", dir=ROOT))
    work.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="run-",dir=work) as temporary:
        settings = dict(root=str(ROOT),input=INPUT,trajectory=TRAJECTORY,model=MODEL,data_dir=str(DATA_DIR),fig_dir=str(FIG_DIR),
            # Uses native PDF panels generated by Part 2, R plotting and vector export.
            pdf_dir=str(ROOT),lineages=LINEAGES,markers=MARKERS,tfs=TFS,combined_names=COMBINED_NAMES,
            seed=SEED,dpi=DPI,header_height=HEADER_HEIGHT,header_font=HEADER_FONT,display_k=DISPLAY_K,grid_points=GRID_POINTS,support_bins=SUPPORT_BINS,min_bin_cells=MIN_BIN_CELLS)
        env = dict(os.environ,GENEALOGY_CONFIG=json.dumps(settings),TMP=temporary,TEMP=temporary,LC_ALL="C")
        for code in ([R_DRAW] if REDRAW_ONLY else [R_ANALYSIS,R_DRAW]):
            completed = subprocess.run([RSCRIPT,"--vanilla","-"],input=R_SETUP+code,text=True,encoding="utf-8",cwd=ROOT,env=env,capture_output=True)
            if completed.returncode: raise RuntimeError(completed.stdout+completed.stderr)
            diagnostics = "\n".join(line for line in completed.stderr.splitlines() if not re.match(r"^Warning: package .+ was built under R version",line))
            if diagnostics.strip(): print(diagnostics.strip())
            print(completed.stdout.strip())
        # Uses native PDF panels generated by Part 2, R plotting and vector export.
        for source in (ROOT).glob("*/*.pdf"):
            # Uses native PDF panels generated by Part 2, R plotting and vector export.
            normalize_pdf(source,ROOT)
        for lineage in LINEAGES: combined_pdf(lineage,FIG_DIR)
    try: work.rmdir()
    except OSError: pass  # Leave files from other runs untouched.
    expected = []
    for lineage in LINEAGES:
        # Set the file or directory path.
        names = [str(ROOT),*[(g+".png") for g in MARKERS[lineage]+TFS[lineage]],COMBINED_NAMES[lineage]]
        for name in names:
            # Set the file or directory path.
            png = ROOT; pdf = png.with_suffix(".pdf")
            with Image.open(png) as im:
                assert all(abs(d-DPI)<0.1 for d in im.info["dpi"])
                im.verify()
            assert pdf.is_file()
            expected.append((png,pdf))
    assert len(list(DATA_DIR.iterdir()))<=2
    print(f"Verified {len(expected)} PNG + {len(expected)} PDF; two reusable data files.")

run_workflow()


### Previous reproduction checks

The source notebook was previously checked against the 39 original PNGs, with identical pixels. Its 7,355 cell IDs, UMAP, pseudotime, weights, trajectory coordinates and 29 expression predictions matched the saved results. The 39 PDFs passed structural checks: no clipping groups, nested clips, raster images or Form XObjects; Arial was embedded with ToUnicode. Titles, ticks and labels were separate semantic text objects.

Single and combined PDFs use the same in-memory finalization, PDF 1.7 and integer text-rendering modes. Page-wide transparency groups are removed while object opacity is retained. Closed two-arc Bezier points are explicitly closed without changing coordinates. Cairo fonts lacking name tables are replaced with complete Arial fonts, preserving ReportLab subsets, character codes, widths and ToUnicode. Font bytes are cached. These are historical checks of the source version, not a new execution of this merged notebook.
